# Encoding in Machine Learning

In this notebook, you will learn:
- Why encoding is needed
- What happens if we skip encoding
- Different encoding methods
- When to use each method
- Easy, step-by-step code examples

## 1) Why do we need encoding?

Most ML algorithms expect numeric input.

But real datasets have categorical text values like:
- city = Delhi, Mumbai, Pune
- education = Bachelors, Masters, PhD
- color = Red, Blue, Green

Models cannot directly calculate distances/weights/probabilities on raw text.
So we convert categories into numbers. This conversion is called **encoding**.

If we skip encoding, training will fail for most algorithms.

In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction import FeatureHasher

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [2]:
# Sample dataset with mixed categorical types
# - gender: nominal (no order)
# - city: nominal (no order)
# - education: ordinal (has natural order)
# - product_id: high-cardinality category
# - purchased: target

df = pd.DataFrame({
    "gender": ["Male", "Female", "Female", "Male", "Female", "Male", "Male", "Female", "Male", "Female"],
    "city": ["Delhi", "Mumbai", "Pune", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
    "education": ["Bachelors", "Masters", "PhD", "Bachelors", "Masters", "School", "PhD", "School", "Bachelors", "Masters"],
    "product_id": ["P101", "P104", "P109", "P101", "P108", "P107", "P106", "P105", "P104", "P110"],
    "purchased": [1, 0, 1, 1, 0, 0, 1, 0, 1, 0]
})

df

,gender,city,education,product_id,purchased
0,Male,Delhi,Bachelors,P101,1
1,Female,Mumbai,Masters,P104,0
2,Female,Pune,PhD,P109,1
3,Male,Delhi,Bachelors,P101,1
4,Female,Pune,Masters,P108,0
5,Male,Mumbai,School,P107,0
6,Male,Delhi,PhD,P106,1
7,Female,Pune,School,P105,0
8,Male,Mumbai,Bachelors,P104,1
9,Female,Delhi,Masters,P110,0


## 2) Label Encoding (good for binary columns)

Label encoding maps categories to integers.

Example:
- Male -> 1
- Female -> 0

Use this mostly for binary categories, not for general nominal columns with many classes.

In [3]:
# Binary mapping for gender (simple and readable)
df_label = df.copy()
df_label["gender_encoded"] = df_label["gender"].map({"Female": 0, "Male": 1})

df_label[["gender", "gender_encoded"]].head()

,gender,gender_encoded
0,Male,1
1,Female,0
2,Female,0
3,Male,1
4,Female,0


## 3) One-Hot Encoding (best for nominal categories)

One-Hot creates separate 0/1 columns for each category.

Why it is useful:
- No false order between categories
- Works well for low/medium-cardinality nominal features

Example:
- city_Delhi, city_Mumbai, city_Pune

In [4]:
# One-hot with pandas
one_hot_city = pd.get_dummies(df["city"], prefix="city")
one_hot_gender = pd.get_dummies(df["gender"], prefix="gender")

pd.concat([df[["city", "gender"]], one_hot_city, one_hot_gender], axis=1).head()

,city,gender,city_Delhi,city_Mumbai,city_Pune,gender_Female,gender_Male
0,Delhi,Male,True,False,False,False,True
1,Mumbai,Female,False,True,False,True,False
2,Pune,Female,False,False,True,True,False
3,Delhi,Male,True,False,False,False,True
4,Pune,Female,False,False,True,True,False


In [ ]:
# One-hot on multiple columns with pandas (simple and readable)
encoded_df = pd.get_dummies(
    df,
    columns=["city", "gender"],
    prefix=["city", "gender"],
    drop_first=True
 )

encoded_df.head()

,city_Delhi,city_Mumbai,city_Pune,gender_Female,gender_Male
0,1,0,0,0,1
1,0,1,0,1,0
2,0,0,1,1,0
3,1,0,0,0,1
4,0,0,1,1,0


## 4) Ordinal Encoding (for ordered categories)

Use this only when categories have natural order.

Example order in education:
School < Bachelors < Masters < PhD

In [6]:
# Manual ordinal mapping (very clear)
edu_order = {"School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}

df_ordinal = df.copy()
df_ordinal["education_encoded"] = df_ordinal["education"].map(edu_order)

df_ordinal[["education", "education_encoded"]].head()

,education,education_encoded
0,Bachelors,1
1,Masters,2
2,PhD,3
3,Bachelors,1
4,Masters,2


In [7]:
# OrdinalEncoder from sklearn
ord_encoder = OrdinalEncoder(categories=[["School", "Bachelors", "Masters", "PhD"]])
edu_encoded = ord_encoder.fit_transform(df[["education"]])

pd.DataFrame({
    "education": df["education"],
    "education_encoded": edu_encoded.flatten().astype(int)
}).head()

,education,education_encoded
0,Bachelors,1
1,Masters,2
2,PhD,3
3,Bachelors,1
4,Masters,2


## 5) Frequency / Count Encoding

Replace each category by how frequently it appears.

Useful for high-cardinality columns where one-hot would create too many columns.

In [8]:
# Frequency encoding on product_id
freq_map = df["product_id"].value_counts(normalize=True)

df_freq = df.copy()
df_freq["product_freq"] = df_freq["product_id"].map(freq_map)

df_freq[["product_id", "product_freq"]].head(10)

,product_id,product_freq
0,P101,0.2
1,P104,0.2
2,P109,0.1
3,P101,0.2
4,P108,0.1
5,P107,0.1
6,P106,0.1
7,P105,0.1
8,P104,0.2
9,P110,0.1


## 6) Target Encoding (powerful, but use carefully)

Target encoding replaces each category with the mean of target for that category.

Important:
- Compute mapping on training data only
- Apply that mapping to validation/test
- Otherwise, data leakage can happen

In [9]:
# Leakage-safe target encoding example on city
X = df[["city"]].copy()
y = df["purchased"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Mapping learned ONLY from train
city_target_map = pd.concat([X_train, y_train], axis=1).groupby("city")["purchased"].mean()
global_mean = y_train.mean()

X_train_te = X_train.copy()
X_test_te = X_test.copy()

X_train_te["city_target_encoded"] = X_train_te["city"].map(city_target_map).fillna(global_mean)
X_test_te["city_target_encoded"] = X_test_te["city"].map(city_target_map).fillna(global_mean)

print("Target means learned from train:\n", city_target_map)
print("\nTrain encoded:\n", X_train_te.head())
print("\nTest encoded:\n", X_test_te.head())

Target means learned from train:
 city
Delhi    0.750000
Pune     0.333333
Name: purchased, dtype: float64

Train encoded:
     city  city_target_encoded
0  Delhi             0.750000
7   Pune             0.333333
2   Pune             0.333333
9  Delhi             0.750000
4   Pune             0.333333

Test encoded:
      city  city_target_encoded
8  Mumbai             0.571429
1  Mumbai             0.571429
5  Mumbai             0.571429


## 7) Hash Encoding (for very high-cardinality features)

Hashing converts category text into a fixed number of numeric columns.

Pros:
- Fixed number of columns
- Memory efficient for large cardinality

Cons:
- Different categories can collide into same hash bucket

In [10]:
# Hash encoding on product_id using sklearn FeatureHasher
hash_input = df["product_id"].apply(lambda x: {x: 1}).tolist()

hasher = FeatureHasher(n_features=4, input_type="dict")
hash_matrix = hasher.transform(hash_input)

hash_df = pd.DataFrame(hash_matrix.toarray(), columns=[f"hash_{i}" for i in range(4)])
pd.concat([df[["product_id"]], hash_df], axis=1).head()

,product_id,hash_0,hash_1,hash_2,hash_3
0,P101,0.0,0.0,-1.0,0.0
1,P104,1.0,0.0,0.0,0.0
2,P109,0.0,0.0,-1.0,0.0
3,P101,0.0,0.0,-1.0,0.0
4,P108,0.0,-1.0,0.0,0.0


## 8) Which encoding should you use?

Quick rule of thumb:
- Binary category -> Label/Binary mapping
- Nominal (low/medium categories) -> One-Hot
- Ordinal (clear order) -> Ordinal Encoding
- High-cardinality -> Frequency or Hashing
- Strong signal categories with enough data -> Target Encoding (with leakage safety)

## 9) Final takeaway

Encoding is required because models need numeric input.
The best method depends on:
- Type of category (nominal/ordinal)
- Number of unique values (cardinality)
- Model type
- Risk of leakage

A good default in many tabular projects:
- One-Hot for low-cardinality nominal columns
- Ordinal for truly ordered columns
- Frequency/Target for high-cardinality columns (carefully)

## 10) Mini practice task

Try this yourself:
1. Add a new categorical column: `department`
2. Apply one-hot encoding on it
3. Compare dataframe shape before and after encoding
4. Try frequency encoding on `department` and compare both outputs

In [11]:
# Optional: quick all-in-one demonstration table
summary_df = df.copy()
summary_df["gender_label"] = summary_df["gender"].map({"Female": 0, "Male": 1})
summary_df["education_ordinal"] = summary_df["education"].map({"School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3})
summary_df["product_freq"] = summary_df["product_id"].map(summary_df["product_id"].value_counts(normalize=True))

one_hot_demo = pd.get_dummies(summary_df[["city"]], prefix="city")
final_demo = pd.concat([summary_df, one_hot_demo], axis=1)

final_demo.head()

,gender,city,education,product_id,purchased,gender_label,education_ordinal,product_freq,city_Delhi,city_Mumbai,city_Pune
0,Male,Delhi,Bachelors,P101,1,1,1,0.2,True,False,False
1,Female,Mumbai,Masters,P104,0,0,2,0.2,False,True,False
2,Female,Pune,PhD,P109,1,0,3,0.1,False,False,True
3,Male,Delhi,Bachelors,P101,1,1,1,0.2,True,False,False
4,Female,Pune,Masters,P108,0,0,2,0.1,False,False,True
